# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a walkthrough for loading and exploring the FAIR^2 dataset using the `mlcroissant` library, focusing on working with Croissant datasets via entity `@id` fields.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the dataset URL (Croissant schema)
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'
# Load the dataset metadata and parse
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset Name: {metadata.name}\n\nDescription: {metadata.description}")

## 2. Data Overview
Display all available record sets and their fields using their `@id`s.

In [ ]:
# List all available record sets by their @id. If none, inform the user.
record_sets = dataset.record_sets
if not record_sets:
    print('This dataset does not contain any record sets defined by @id.')
else:
    for rs in record_sets:
        print(f"Record set @id: {rs['@id']}")
        print(f"  Name: {rs.get('name', '[No name]')}")
        print("  Fields:")
        for field in rs.get('fields', []):
            print(f"    - @id: {field['@id']} | Name: {field.get('name', '[No name]')}")
        print()
# If available, display the full list of record sets and fields as a DataFrame
rs_list = []
for rs in record_sets:
    fields = rs.get('fields', [])
    for field in fields:
        rs_list.append({
            'record_set_id': rs['@id'],
            'record_set_name': rs.get('name', None),
            'field_id': field['@id'],
            'field_name': field.get('name', None)
        })
if rs_list:
    overview_df = pd.DataFrame(rs_list)
    display(overview_df)

## 3. Data Extraction
Load data from a specific record set using its `@id`. We'll extract all available record sets into pandas DataFrames for further analysis.

**Note:** If the dataset doesn't have any record sets, this code will not execute extraction.

In [ ]:
# Build a list of record set @ids
record_set_ids = [rs['@id'] for rs in record_sets]

dataframes = {}
for record_set_id in record_set_ids:
    # Load all records for this record set
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded record set '@id': {record_set_id}")
        print(f"Columns: {df.columns.tolist()}")
        display(df.head())
    except Exception as e:
        print(f"Could not load records for '{record_set_id}': {e}")

# For demonstration, select the first available record set (if any)
if record_set_ids:
    selected_record_set_id = record_set_ids[0]
    sample_df = dataframes[selected_record_set_id]
    print(f"Columns in record set {selected_record_set_id}: {sample_df.columns.tolist()}")
    display(sample_df.head())
else:
    print('No record sets available for extraction.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data by key attributes.
All references to fields and record sets use their `@id`s as per Croissant best practices.

In [ ]:
# Example EDA: Filter and Normalize Numeric Field, and Group by Field
if record_set_ids:
    df = dataframes[selected_record_set_id]
    # Find a numeric field (by column dtype or known @id)
    numeric_field_id = None
    group_field_id = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
    # Fallback: Use first column if none detected (for demonstration purposes only)
    if not numeric_field_id and len(df.columns) > 0:
        numeric_field_id = df.columns[0]
    # Try to find a suitable group field (string column)
    for col in df.columns:
        if pd.api.types.is_string_dtype(df[col]):
            group_field_id = col
            break
    print(f"Numeric field chosen (by @id): {numeric_field_id}")
    print(f"Group field chosen (by @id): {group_field_id}")

    # Example threshold, mean, etc. (use 10 as threshold when possible)
    try:
        threshold = 10
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with '{numeric_field_id}' > {threshold}:")
        display(filtered_df.head())
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        )
        print(f"Normalized '{numeric_field_id}' for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        if group_field_id is not None:
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"Grouped data by '{group_field_id}':")
            display(grouped_df.head())
    except Exception as ex:
        print(f"EDA example could not be performed: {ex}")
else:
    print('EDA skipped: No record sets with data available.')

## 5. Visualization
Visualize data distributions or relationships using the selected record set and field `@id`s.

Example: A histogram of the selected numeric field and a boxplot by group (if possible).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_ids and numeric_field_id in df:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Histogram of '{numeric_field_id}'")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

    if group_field_id is not None:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"Boxplot of '{numeric_field_id}' by '{group_field_id}'")
        plt.xticks(rotation=45)
        plt.show()
else:
    print('No numeric field available for visualization.')

## 6. Conclusion
This notebook provided a structured workflow for exploring Croissant datasets using the `mlcroissant` library, referencing all entities via their `@id` fields for traceability. You can extend this approach to more detailed data processing and domain-specific analysis.